In [3]:
import os
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader  # 修正
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from torch.utils.tensorboard import SummaryWriter
from sklearn.model_selection import KFold

if torch.cuda.is_available():
    print("GPU is available")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available, using CPU")
    
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")


# データフォルダ
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/Defectlabel4x4/DefectLabels_4x4_test1"

# 座標データのロード
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# エッジ情報を読み込む
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# データとラベルファイルを対応付け
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]


# データとラベルのペアを取得する関数
def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)  # 欠陥なしデータの場合
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"無効なファイル名の形式: {file_name}")
        return None

def prepare_data(pairs):
    sampled_data = []
    sampled_labels = []
    
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # データとラベルを読み込む
        try:
            values = np.load(data_file_path)[:3654]
            label = np.load(label_file_path)[:3654]
        except Exception as e:
            print(f"Error loading data: {e}")
            continue
        
        # 座標データと応力データを結合してノード特徴量を作成
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)
    
    if len(sampled_data) == 0 or len(sampled_labels) == 0:
        print("No valid data found in the pairs.")
        return None

    # データを結合しテンソルに変換
    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.float).to(device)

    return x, y


def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"無効なファイル名の形式: {file_name}")
        return None

data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# 有効なペアを取得し、欠陥なしデータを追加
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]
defect_free_pair = ("/home/nishioka/GNN/BasicdataforGNN/Normalized1_nodefect_ElNOD.npy",
                    "/home/nishioka/GNN/BasicdataforGNN/DefectLabel_nodefect.npy")
for _ in range(100):
    valid_pairs.append(defect_free_pair)

def count_pair_occurrences(pairs, target_pair):
    return pairs.count(target_pair)

pair_count = count_pair_occurrences(valid_pairs, defect_free_pair)
print(f"\nDefect-free data occurrences in train_pairs: {pair_count} (expected: 100)")

# データを準備
all_x, all_y = prepare_data(valid_pairs)

# 全体データからの DataLoader 用のサンプルリスト作成
all_data_list = []
num_all_samples = all_x.shape[0] // 3654
for i in range(num_all_samples):
    all_data_list.append(Data(x=all_x[i * 3654:(i + 1) * 3654], edge_index=edge_index, y=all_y[i * 3654:(i + 1) * 3654]))

# KFoldの設定
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# 各フォールドの結果を保存するリスト
fold_train_losses = []
fold_val_losses = []
fold_test_losses = []
fold_rmse = []
fold_mae = []
fold_r2 = []


hidden_channels = 128
learning_rate = 0.0005
batch_size = 32
epochs = 1500
weight_decay = 5e-4
patience = 50


# Xavier初期化関数
def init_weights(m):
    if isinstance(m, torch.nn.Linear):
        torch.nn.init.xavier_uniform_(m.weight)
    elif isinstance(m, GCNConv):
        torch.nn.init.xavier_uniform_(m.lin.weight)

class GCNModel(torch.nn.Module):
    def __init__(self, hidden_channels=hidden_channels):
        super(GCNModel, self).__init__()
        self.conv1 = GCNConv(4, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)
        self.conv3 = GCNConv(hidden_channels * 2, hidden_channels * 4)
        self.conv4 = GCNConv(hidden_channels * 4, hidden_channels * 2)
        self.conv5 = GCNConv(hidden_channels * 2, hidden_channels)
        self.fc_defect = nn.Linear(hidden_channels, 1)
        self.dropout = nn.Dropout(p=0.2)  # ドロップアウト率

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv3(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv4(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv5(x, edge_index))
        x = self.fc_defect(x)
        return torch.sigmoid(x)

# KFoldでの交差検証
for fold, (train_idx, val_idx) in enumerate(kf.split(all_data_list), start=1):
    print(f"\nStarting fold {fold} of {n_splits}...")
    
    # インデックスに基づいてデータを分割
    train_data_list = [all_data_list[i] for i in train_idx]
    val_data_list = [all_data_list[i] for i in val_idx]
    # test_data_list = [all_data_list[i] for i in test_idx]

    # データローダーの作成
    train_loader = DataLoader(train_data_list, batch_size=8, shuffle=True)
    val_loader = DataLoader(val_data_list, batch_size=8, shuffle=False)
    # test_loader = DataLoader(test_data_list, batch_size=8, shuffle=False)

    # モデル、オプティマイザ、損失関数の定義
    model = GCNModel(hidden_channels=hidden_channels).to(device)
    model.apply(init_weights)  # Xavier初期化
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

class EarlyStopping:
    def __init__(self, patience=patience, verbose=False, path='/home/nishioka/GNN/checkpoint1.pt', delta=0, trace_func=print):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.Inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func
    
    def __call__(self, val_loss, model):
        score = -val_loss
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            # if self.verbose:
            #     self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0
    
    def save_checkpoint(self, val_loss, model):
        # if self.verbose:
        #     self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

# EarlyStopping クラスのインスタンス作成
early_stopping = EarlyStopping(patience=100, verbose=True, path=f'/home/nishioka/GNN/checkpoint_fold_{fold}_{timestamp}.pth')

# 各フォールドごとの学習と検証
train_losses = []
val_losses = []
test_losses = []
start_time = time.time()

for epoch in range(1, 2001):  # 最大200エポック
    model.train()
    total_loss = 0

    for batch in train_loader:
        batch.x = batch.x.to(device)
        batch.y = batch.y.to(device)
        batch.edge_index = batch.edge_index.to(device)
        optimizer.zero_grad()
        out = model(batch)

        # 重み付きHuber損失を使用
        delta = 0.1
        huber_loss = torch.where(
            torch.abs(out - batch.y.view(-1, 1)) < delta,
            0.5 * (out - batch.y.view(-1, 1))**2,
            delta * torch.abs(out - batch.y.view(-1, 1)) - 0.5 * delta**2
        )
        weights = torch.where(batch.y > 0, torch.tensor(1000.0).to(device), torch.tensor(1.0).to(device))
        loss = (weights * huber_loss).mean()

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    train_losses.append(total_loss / len(train_loader))

    # 検証ステップ
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            batch.x = batch.x.to(device)
            batch.y = batch.y.to(device)
            batch.edge_index = batch.edge_index.to(device)
            out = model(batch)

            # 重み付きHuber損失を使用
            weights = torch.where(batch.y > 0, torch.tensor(800.0).to(device), torch.tensor(1.0).to(device))
            loss = (weights * huber_loss).mean()
            val_loss += loss.item()

    val_losses.append(val_loss / len(val_loader))

    # EarlyStoppingに現在のバリデーション損失とモデルを渡す
    early_stopping(val_loss / len(val_loader), model)

    # EarlyStoppingがTrueなら訓練を終了
    if early_stopping.early_stop:
        print(f"Early stopping at epoch {epoch}")
        break

        # 1, 10, 30, 50, 100, 150, 200, 250... のタイミングで結果を出力
    if epoch == 1 or epoch == 10 or epoch == 30 or (epoch % 50 == 0):
        elapsed_time = time.time() - start_time
        print(f'Epoch {epoch}, Train Loss: {train_losses[-1]:.4f}, Val Loss: {val_losses[-1]:.4f}, Time: {elapsed_time:.2f}s')


# # テストデータでの評価
# test_loss = 0
# test_preds = []
# test_targets = []
# with torch.no_grad():
#     for batch in test_loader:
#         batch.x = batch.x.to(device)
#         batch.y = batch.y.to(device)
#         batch.edge_index = batch.edge_index.to(device)
#         out = model(batch)
#         # loss = loss_fn(out, batch.y.view(-1, 1))

#         weights = torch.where(batch.y > 0, torch.tensor(5.0).to(device), torch.tensor(1.0).to(device))
#         loss = (weights * (out - batch.y.view(-1, 1))**2).mean()
        
#         test_loss += loss.item()
#         test_preds.append(out.cpu().numpy())
#         test_targets.append(batch.y.cpu().numpy())

# test_loss /= len(test_loader)
# test_preds = np.concatenate(test_preds)
# test_targets = np.concatenate(test_targets)

# RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(test_targets, test_preds))

# MAE (Mean Absolute Error)
mae = mean_absolute_error(test_targets, test_preds)

# R2 Score
r2 = r2_score(test_targets, test_preds)

print(f'Test Loss: {test_loss:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2 Score: {r2:.4f}')

# TensorBoardに損失を記録
writer.add_scalar('Loss/train', train_losses[-1], epoch)
writer.add_scalar('Loss/val', val_losses[-1], epoch)
        
# 学習時間の計測終了
elapsed_time = time.time() - start_time
print(f"Total training time: {elapsed_time:.2f} seconds")

# モデル、学習結果の保存
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gcn_model_final_{timestamp}.pth')

# 学習時間をファイルに保存
with open(f'/home/nishioka/GNN/GNNmodelcsv/training_time_{timestamp}.txt', 'w') as f:
    f.write(f"Total training time: {elapsed_time:.2f} seconds\n")

# 損失とその他の指標をCSVに保存
loss_data = pd.DataFrame({
    'Epoch': range(1, len(train_losses) + 1),
    'Training Loss': train_losses,
    'Validation Loss': val_losses
})
loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/loss_data_{timestamp}.csv', index=False)


# 損失をプロット
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss', color='blue')  # train_lossesの長さに合わせる
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss', color='orange')  # val_lossesの長さに合わせる
plt.yscale('log')  # 対数スケール
plt.xlabel('Epoch')
plt.ylabel('Loss (Log Scale)')
plt.title(f'Training and Validation Loss {type(model).__name__} - ({timestamp})')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/loss_fig_{type(model).__name__}_{timestamp}.png')
plt.show()

# Create a figure for the plot and the table
fig, ax = plt.subplots(figsize=(10, 6))

# Remove the axes (the empty graph)
ax.axis('off')

# Data for the table
table_data = [
    ["Model", type(model).__name__],
    ["Learning Rate", learning_rate],
    ["Batch Size", batch_size],
    ["Epochs", epochs],
    ["Weight Decay", weight_decay],
    ["Patience", patience],
    ["Training Samples", num_train_samples],
    ["Validation Samples", num_val_samples],
    ["Test Samples", num_test_samples]
]

# Create the table without any plot axes
table = ax.table(cellText=table_data, colLabels=["Parameter", "Value"], loc="center", cellLoc="center")

# Adjust column widths and the table position
table.scale(1.5, 1.5)  # Scale width (1.5 times wider) and height (1.5 times taller)
table.set_fontsize(12)  # Set table font size

# Save the figure
plt.subplots_adjust(left=0.1, bottom=0.2)  # Ensure there’s enough space at the bottom for the table
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/parameter_table_{timestamp}.png')
plt.show()


# モデルとハイパーパラメータをJSONに保存する例
model_params = {
    'hidden_channels': hidden_channels,
    'learning_rate': learning_rate,
    'batch_size': batch_size,
    'epochs': epochs,
    'weight_decay': weight_decay,
    'patience': patience
}
with open(f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_params_{timestamp}.json', 'w') as f:
    json.dump(model_params, f, indent=4)


# 損失とハイパーパラメータ情報をCSVに保存
loss_data = pd.DataFrame({
    'Epoch': range(1, epochs + 1),
    'Training Loss': train_losses,
    'Validation Loss': val_losses
})
loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/loss_data_{test_loss:.4f}_{timestamp}.csv', index=False)

# ハイパーパラメータのCSV
hyperparameter_data = pd.DataFrame(table_data, columns=["Parameter", "Value"])
hyperparameter_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/hyperparameters_{test_loss:.4f}_{timestamp}.csv', index=False)

# 結果を保存
test_loss_data = pd.DataFrame({
    'Test Loss': [test_loss],
    'RMSE': [rmse],
    'MAE': [mae],
    'R2 Score': [r2]
})
test_loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/test_loss_{test_loss:.4f}_{timestamp}.csv', index=False)

# ファイル名にタイムスタンプを追加

test_loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/test_loss_{test_loss:.4f}_{timestamp}.csv', index=False)

# モデルの保存（ファイル名にタイムスタンプを追加）
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_{test_loss:.4f}_{timestamp}.pth')
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodelweights/{type(model).__name__}_weights_{test_loss:.4f}_{timestamp}.pth')

# TensorBoardのWriterを閉じる
writer.close()

# Improved Final summary after the training loop
# def print_summary(epoch, early_stop):
print("\nTraining Summary:")
print(f"Total Epochs Run: {epoch}")
if early_stopping.early_stop:
    print(f"Early stopping at epoch {epoch}")
else:
    print("Training continued without early stopping.")
print(f"Best Epoch (Lowest Validation Loss): {epoch}")
print(f"Final Training Loss: {train_losses[-1]:.4f}")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")
print(f"Final Learning Rate: {learning_rate}")
print(f"Training Samples: {num_train_samples}")
print(f"Validation Samples: {num_val_samples}")
print(f"Test Samples: {num_test_samples}")
print(f"Total Training Time: {elapsed_time:.2f} seconds")

GPU is available
Using device: NVIDIA GeForce RTX 3090

Defect-free data occurrences in train_pairs: 100 (expected: 100)

Starting fold 1 of 5...

Starting fold 2 of 5...

Starting fold 3 of 5...

Starting fold 4 of 5...

Starting fold 5 of 5...
Epoch 1, Train Loss: 0.0234, Val Loss: 0.0061, Time: 7.48s
Epoch 10, Train Loss: 0.0081, Val Loss: 0.0052, Time: 74.59s
Epoch 30, Train Loss: 0.0081, Val Loss: 0.0061, Time: 223.94s
Epoch 50, Train Loss: 0.0081, Val Loss: 0.0070, Time: 373.46s
Epoch 100, Train Loss: 0.0081, Val Loss: 0.0070, Time: 747.57s
Epoch 150, Train Loss: 0.0058, Val Loss: 0.0045, Time: 1122.22s
Epoch 200, Train Loss: 0.0054, Val Loss: 0.0042, Time: 1496.78s
Early stopping at epoch 244


NameError: name 'test_loader' is not defined

In [4]:
# RMSE (Root Mean Squared Error)
rmse = np.sqrt(mean_squared_error(test_targets, test_preds))

# MAE (Mean Absolute Error)
mae = mean_absolute_error(test_targets, test_preds)

# R2 Score
r2 = r2_score(test_targets, test_preds)

print(f'Test Loss: {test_loss:.4f}, RMSE: {rmse:.4f}, MAE: {mae:.4f}, R2 Score: {r2:.4f}')

# TensorBoardに損失を記録
writer.add_scalar('Loss/train', train_losses[-1], epoch)
writer.add_scalar('Loss/val', val_losses[-1], epoch)
        
# 学習時間の計測終了
elapsed_time = time.time() - start_time
print(f"Total training time: {elapsed_time:.2f} seconds")

# モデル、学習結果の保存
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gcn_model_final_{timestamp}.pth')

# 学習時間をファイルに保存
with open(f'/home/nishioka/GNN/GNNmodelcsv/training_time_{timestamp}.txt', 'w') as f:
    f.write(f"Total training time: {elapsed_time:.2f} seconds\n")

# 損失とその他の指標をCSVに保存
loss_data = pd.DataFrame({
    'Epoch': range(1, len(train_losses) + 1),
    'Training Loss': train_losses,
    'Validation Loss': val_losses
})
loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/loss_data_{timestamp}.csv', index=False)


# 損失をプロット
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, label='Training Loss', color='blue')  # train_lossesの長さに合わせる
plt.plot(range(1, len(val_losses) + 1), val_losses, label='Validation Loss', color='orange')  # val_lossesの長さに合わせる
plt.yscale('log')  # 対数スケール
plt.xlabel('Epoch')
plt.ylabel('Loss (Log Scale)')
plt.title(f'Training and Validation Loss {type(model).__name__} - ({timestamp})')
plt.legend()
plt.grid(True, which="both", ls="--")
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/loss_fig_{type(model).__name__}_{timestamp}.png')
plt.show()

# Create a figure for the plot and the table
fig, ax = plt.subplots(figsize=(10, 6))

# Remove the axes (the empty graph)
ax.axis('off')

# Data for the table
table_data = [
    ["Model", type(model).__name__],
    ["Learning Rate", learning_rate],
    ["Batch Size", batch_size],
    ["Epochs", epochs],
    ["Weight Decay", weight_decay],
    ["Patience", patience],
    ["Training Samples", num_train_samples],
    ["Validation Samples", num_val_samples],
    ["Test Samples", num_test_samples]
]

# Create the table without any plot axes
table = ax.table(cellText=table_data, colLabels=["Parameter", "Value"], loc="center", cellLoc="center")

# Adjust column widths and the table position
table.scale(1.5, 1.5)  # Scale width (1.5 times wider) and height (1.5 times taller)
table.set_fontsize(12)  # Set table font size

# Save the figure
plt.subplots_adjust(left=0.1, bottom=0.2)  # Ensure there’s enough space at the bottom for the table
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/parameter_table_{timestamp}.png')
plt.show()


# モデルとハイパーパラメータをJSONに保存する例
model_params = {
    'hidden_channels': hidden_channels,
    'learning_rate': learning_rate,
    'batch_size': batch_size,
    'epochs': epochs,
    'weight_decay': weight_decay,
    'patience': patience
}
with open(f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_params_{timestamp}.json', 'w') as f:
    json.dump(model_params, f, indent=4)


# 損失とハイパーパラメータ情報をCSVに保存
loss_data = pd.DataFrame({
    'Epoch': range(1, epochs + 1),
    'Training Loss': train_losses,
    'Validation Loss': val_losses
})
loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/loss_data_{test_loss:.4f}_{timestamp}.csv', index=False)

# ハイパーパラメータのCSV
hyperparameter_data = pd.DataFrame(table_data, columns=["Parameter", "Value"])
hyperparameter_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/hyperparameters_{test_loss:.4f}_{timestamp}.csv', index=False)

# 結果を保存
test_loss_data = pd.DataFrame({
    'Test Loss': [test_loss],
    'RMSE': [rmse],
    'MAE': [mae],
    'R2 Score': [r2]
})
test_loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/test_loss_{test_loss:.4f}_{timestamp}.csv', index=False)

# ファイル名にタイムスタンプを追加

test_loss_data.to_csv(f'/home/nishioka/GNN/GNNmodelcsv/test_loss_{test_loss:.4f}_{timestamp}.csv', index=False)

# モデルの保存（ファイル名にタイムスタンプを追加）
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/{type(model).__name__}_{test_loss:.4f}_{timestamp}.pth')
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodelweights/{type(model).__name__}_weights_{test_loss:.4f}_{timestamp}.pth')

# TensorBoardのWriterを閉じる
writer.close()

# Improved Final summary after the training loop
# def print_summary(epoch, early_stop):
print("\nTraining Summary:")
print(f"Total Epochs Run: {epoch}")
if early_stopping.early_stop:
    print(f"Early stopping at epoch {epoch}")
else:
    print("Training continued without early stopping.")
print(f"Best Epoch (Lowest Validation Loss): {epoch}")
print(f"Final Training Loss: {train_losses[-1]:.4f}")
print(f"Best Validation Loss: {best_val_loss:.4f}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R2 Score: {r2:.4f}")
print(f"Final Learning Rate: {learning_rate}")
print(f"Training Samples: {num_train_samples}")
print(f"Validation Samples: {num_val_samples}")
print(f"Test Samples: {num_test_samples}")
print(f"Total Training Time: {elapsed_time:.2f} seconds")

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.